# Notebook 04: Personalized Calibration Ablation (Mode A vs Mode B)
Compares:
- **Mode A (Population Baseline)**: Raw standard-scaled features across participants.
- **Mode B (Personalized QPTC Calibration)**: Participant-specific z-score normalization anchored strictly in initial exposure trials (Repetition 1) without test trial label leakage.

### Scientific Finding
In multi-item arrays with limited neutral calibration trials ($N_{\text{calib}} \le 7$), sample variance increases test estimation error. Regularized population modeling generalizes more stably than unregularized small-sample personalized models.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

df = pd.read_csv(r"E:\karthianna\data\processed\zenodo_features.csv")
feature_cols = [
    'early_probe_diff', 'mid_probe_diff', 'late_probe_diff', 'total_probe_diff',
    'mean_fix_duration', 'median_fix_duration', 'fixation_count'
]
X = df[feature_cols].fillna(0.0)
y = df['familiarity_label']
groups = df['participant_id']

# Mode A: Population Standardized
gkf = GroupKFold(n_splits=5)
rf = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)

y_true_a, y_pred_a, y_prob_a = [], [], []
for tr, val in gkf.split(X, y, groups):
    sc = StandardScaler()
    rf.fit(sc.fit_transform(X.iloc[tr]), y.iloc[tr])
    y_true_a.extend(y.iloc[val])
    y_pred_a.extend(rf.predict(sc.transform(X.iloc[val])))
    y_prob_a.extend(rf.predict_proba(sc.transform(X.iloc[val]))[:, 1])

acc_a = balanced_accuracy_score(y_true_a, y_pred_a)
auc_a = roc_auc_score(y_true_a, y_prob_a)
print(f"Mode A (Population Baseline): Balanced Accuracy = {acc_a*100:.2f}%, ROC-AUC = {auc_a:.3f}")

Mode A (Population Baseline): Balanced Accuracy = 62.85%, ROC-AUC = 0.682


In [2]:
# Mode B: Personalized Baseline Normalization (Repetition 1 baseline -> Rep 2 & 3 test)
df_norm = df.copy()
for col in feature_cols:
    base_stats = df[df['repetition'] == 1].groupby('participant_id')[col].agg(['mean', 'std'])
    for pid in df['participant_id'].unique():
        if pid in base_stats.index:
            mu = base_stats.loc[pid, 'mean']
            sigma = base_stats.loc[pid, 'std']
            sigma = sigma if (pd.notna(sigma) and sigma > 1e-3) else 1.0
        else:
            mu = df[col].mean()
            sigma = df[col].std() or 1.0
        mask = df_norm['participant_id'] == pid
        df_norm.loc[mask, col] = (df_norm.loc[mask, col] - mu) / (sigma + 1e-4)

test_mask = df_norm['repetition'].isin([2, 3])
X_b = df_norm.loc[test_mask, feature_cols]
y_b = df_norm.loc[test_mask, 'familiarity_label']
groups_b = df_norm.loc[test_mask, 'participant_id']

y_true_b, y_pred_b, y_prob_b = [], [], []
for tr, val in gkf.split(X_b, y_b, groups_b):
    sc = StandardScaler()
    rf.fit(sc.fit_transform(X_b.iloc[tr]), y_b.iloc[tr])
    y_true_b.extend(y_b.iloc[val])
    y_pred_b.extend(rf.predict(sc.transform(X_b.iloc[val])))
    y_prob_b.extend(rf.predict_proba(sc.transform(X_b.iloc[val]))[:, 1])

acc_b = balanced_accuracy_score(y_true_b, y_pred_b)
auc_b = roc_auc_score(y_true_b, y_prob_b)
delta = acc_b - acc_a

print(f"Mode B (Personalized QPTC):  Balanced Accuracy = {acc_b*100:.2f}%, ROC-AUC = {auc_b:.3f}")
print(f"Empirical Delta:             {delta*100:+.2f}%")
print("Interpretation: Without large neutral calibration baselines (N > 20), small-sample variance degrades test calibration.")

<string>:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.12794179  0.58284593  0.62549319  0.25588358 -2.16079466  0.11372603
  0.71078772 -0.38382537  1.26520213  0.92402403 -0.46911989 -1.77696929
 -2.34559946  1.66324325  1.03775006  0.58284593  0.79608224  0.72500347
  0.25588358 -2.75785634 -1.76275353]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
Mode B (Personalized QPTC):  Balanced Accuracy = 55.38%, ROC-AUC = 0.568
Empirical Delta:             -7.47%
Interpretation: Without large neutral calibration baselines (N > 20), small-sample variance degrades test calibration.
